## Same as DESI, we run fiber assignment for one tile at a time, which is slow but may not meet OOM issue. --06-17-2026

In [1]:
import os, sys, json
import glob
import numpy as np
import time, gc
import argparse
from astropy.table import Table, vstack
import multiprocessing
import contextlib
from numpy.random import Generator, PCG64
from collections import defaultdict
sys.path.append("/home/zjding/installed_packages/JUST_fiberassign/py/")
from utils import (
    get_fiberpos, write_fba_onetile,
    mask_targets_in_tile,
    find_targets_in_one_tile,
    find_neighboring_fibers,
    find_collided_pairs_in_one_tile,
    find_overlapped_tile_groups
    )
from parameters import (
    COLLISION_SEPARATION_ARCSEC,
    COLLISION_SEPARATION_DEG,
    COLLISION_SEPARATION_MM,
    TILE_INNER_RADIUS_DEG,
    TILE_OUTER_RADIUS_DEG,
    R_PATROL_DEG)
from network_flow import (
    solve_tile_group,
    aggregate_group_assignments_with_pairwise_repair
    )
import network_flow as _network_flow_module
import fitsio
import healpy as hp
import matplotlib.pyplot as plt

In [2]:
def _log(msg):
    print(msg, flush=True)

def _fits_path_for_tile(out_dir, tile_id):
    return os.path.join(out_dir, f"fba_tile_{int(tile_id)}.fits")


def _checkpoint_path(out_dir):
    return os.path.join(out_dir, "fba_checkpoint.json")


def _list_completed_tiles(out_dir):
    completed = set()
    for path in glob.glob(os.path.join(out_dir, "fba_tile_*.fits")):
        base = os.path.basename(path)
        try:
            completed.add(int(base[len("fba_tile_") : -len(".fits")]))
        except ValueError:
            continue
    return completed


def _save_checkpoint(path, state):
    tmp_path = f"{path}.tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, path)


def _healpix_ids_for_disc(ra_deg, dec_deg, radius_deg, nside, nest=False):
    """HEALPix pixel IDs that can contain sources within radius_deg of (ra, dec)."""
    vec = hp.ang2vec(ra_deg, dec_deg, lonlat=True)
    return np.unique(
        hp.query_disc(nside, vec, np.radians(radius_deg), nest=nest, inclusive=True)
    ).astype(np.int64)



def load_galaxies_from_mtlpix(
    ra_deg,
    dec_deg,
    odir_mtl,
    radius_deg,
    nside=32,
    nest=False,
):
    """
    Load galaxies from mtl_nside*/mtl_healpix_*.fits for HEALPix pixels
    overlapping a disc of radius_deg around (ra_deg, dec_deg).
    """
    pix_ids = _healpix_ids_for_disc(ra_deg, dec_deg, radius_deg, nside, nest=nest)
    cats = []
    for pix_id in pix_ids:
        fpath = os.path.join(odir_mtl, f"mtl_healpix_{int(pix_id):05d}.fits")
        if os.path.isfile(fpath):
            cats.append(Table.read(fpath))
    if not cats:
        return Table(
            names=["TARGETID", "RA", "DEC", "PRIORITY", "SUBPRIORITY", "HEALPIXID"],
            dtype=[np.int64, np.float64, np.float64, np.float64, np.float64, np.int64],
        )

    return vstack(cats, metadata_conflicts="silent")


def _degrade_mtl_priorities_on_disk(mtl_dir, assigned_target_ids, priority_degraded, healpix_ids):
    """Set PRIORITY=priority_degraded for assigned targets in HEALPix MTL FITS files."""
    if not assigned_target_ids:
        return 0

    n_updated = 0
    for pix_id in np.unique(healpix_ids):
        fpath = os.path.join(mtl_dir, f"mtl_healpix_{int(pix_id):05d}.fits")
        if not os.path.isfile(fpath):
            continue
        cat_pix = Table.read(fpath)
        mask = np.isin(cat_pix["TARGETID"], assigned_target_ids)
        if not np.any(mask):
            continue
        cat_pix["PRIORITY"][mask] = priority_degraded
        cat_pix.write(fpath, overwrite=True)
        n_updated += int(mask.sum())
    return n_updated


class _SerialEvalPool:
    def __init__(self, processes=None, **kwargs):
        self.processes = processes or 1

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        return False

    def starmap(self, func, iterable):
        return [func(*args) for args in iterable]

    def map(self, func, iterable):
        return [func(item) for item in iterable]


@contextlib.contextmanager
def _avoid_nested_pool_in_solve():
    original_pool = _network_flow_module.multiprocessing.Pool
    _network_flow_module.multiprocessing.Pool = _SerialEvalPool
    try:
        yield
    finally:
        _network_flow_module.multiprocessing.Pool = original_pool

In [3]:
rmagcut = 20.5
Npasses = 3
ra0 = 30.0
ra1 = 40.0
dec0 = 10.0
dec1 = 20.0
n_workers = 8
eval_workers = 1
max_iterations = 3
rand_seed = 3
#no_resume = args.no_resume

pool_workers = max(1, n_workers - 2) if n_workers > 2 else n_workers

rng = Generator(PCG64(seed=rand_seed))
# Constants
_log(
    f"rand_seed={rand_seed}, pool_workers={pool_workers}, "
    f"eval_workers={eval_workers}"
)
_log(f"Patrol radius: {R_PATROL_DEG:.6f} degrees")
_log(f"Collision separation: {COLLISION_SEPARATION_ARCSEC} arcsec = {COLLISION_SEPARATION_DEG:.6f} deg = {COLLISION_SEPARATION_MM:.3f} mm")

# Get fiber positions and set random seed
fiberpos_xy = get_fiberpos()
N_fibers = fiberpos_xy.shape[0]
_log(f"Number of fibers: {N_fibers}")

# Define overflow cost
COST_OVERFLOW = 10.0

rand_seed=3, pool_workers=6, eval_workers=1
Patrol radius: 0.013021 degrees
Collision separation: 15.625 arcsec = 0.004340 deg = 2.000 mm
Number of fibers: 2184


In [4]:
dir_root = "/home/zjding/fiberassignment/JUST/BGS_mock/Junyu_mock/"
ifile = dir_root + f"data/lightcone_ra_0_90_dec_0_90_rmagcut{rmagcut}.fits"
columns = ["ra", "dec", "idx"]
input_cat = fitsio.read(ifile, columns=columns)
mask = (input_cat["ra"]>ra0)&(input_cat["ra"]<ra1)&(input_cat["dec"]>dec0)&(input_cat["dec"]<dec1)
input_cat = input_cat[mask]

gal_MTL = Table()
priority_initial = 4.0      # initial priority for all targets
priority_degraded = 1.0     # degraded priority for targets that have been fiber assigned
columns=["TARGETID", "RA", "DEC", "PRIORITY", "SUBPRIORITY"]

for col in columns:
    if col == "TARGETID":
        gal_MTL[col] = input_cat["idx"]
    elif col == "RA":
        gal_MTL[col] = input_cat["ra"]
    elif col == "DEC":
        gal_MTL[col] = input_cat["dec"]
    elif col == "PRIORITY":
        gal_MTL[col] = np.ones(len(input_cat), dtype=float)*priority_initial
    elif col == "SUBPRIORITY":
        gal_MTL[col] = rng.random(len(input_cat))
    else:
        gal_MTL[col] = input_cat[col]
del input_cat; gc.collect()

0

In [5]:
# load tiles
ifile = dir_root + "fba/input/tiles_4passes.fits"
tile_data = Table.read(ifile)
tile_ra0, tile_ra1 = ra0 + TILE_OUTER_RADIUS_DEG, ra1 - TILE_OUTER_RADIUS_DEG
tile_dec0, tile_dec1 = dec0 + TILE_OUTER_RADIUS_DEG, dec1 - TILE_OUTER_RADIUS_DEG

mask = (tile_data["RA"]>tile_ra0)&(tile_data["RA"]<tile_ra1)&(tile_data["DEC"]>tile_dec0)&(tile_data["DEC"]<tile_dec1)
mask &= (tile_data["PASS"]<Npasses)   # PASS starts from 0
tiles_sub = tile_data[mask]
N_tiles = len(tiles_sub)

neighboring_fiber_pairs = find_neighboring_fibers(fiberpos_xy, patrol_center_separation=12.0)
_log(f"There are {len(neighboring_fiber_pairs)} paired fibers in one tile")

out_dir = dir_root + f"fba/output/{Npasses}passes/{N_tiles}tiles_{ra0:.1f}ra{ra1:.1f}_{dec0:.1f}dec{dec1:.1f}/seed{rand_seed}/"
os.makedirs(out_dir, exist_ok=True)

There are 6312 paired fibers in one tile


In [6]:
## split MTL catalog into different files with pixel IDs
nside = 32
nest = False  # RING ordering (healpy default)

npix = hp.nside2npix(nside)
pix_area_deg2 = hp.nside2pixarea(nside, degrees=True)
print(f"HEALPix nside={nside}: {npix} pixels, ~{np.sqrt(pix_area_deg2):.2f} deg/pixel side")

# Assign each galaxy to a HEALPix pixel from (ra, dec) in degrees
ra = np.asarray(gal_MTL["RA"], dtype=np.float64)
dec = np.asarray(gal_MTL["DEC"], dtype=np.float64)
pix = hp.ang2pix(nside, ra, dec, lonlat=True, nest=nest)

gal_MTL["HEALPIXID"] = pix.astype(np.int64)
print(f"Assigned {len(gal_MTL)} galaxies to {len(np.unique(pix))} non-empty pixels")

# Group galaxies by HEALPix pixel (sorted slices, no full-table copy per pixel)
sort_idx = np.argsort(pix, kind="stable")
pix_sorted = pix[sort_idx]
unique_pix, start_idx, counts = np.unique(
    pix_sorted, return_index=True, return_counts=True
)

pixel_cats = {}
for pix_id, start, count in zip(unique_pix, start_idx, counts):
    rows = sort_idx[start : start + count]
    pixel_cats[int(pix_id)] = gal_MTL[rows]

print(f"Built pixel_cats with {len(pixel_cats)} entries")
print(
    "Galaxies per pixel: "
    f"min={counts.min()}, median={np.median(counts):.0f}, max={counts.max()}"
)

# Example: inspect one populated pixel
example_pix = int(unique_pix[counts.argmax()])
print(f"Example pixel {example_pix}: {len(pixel_cats[example_pix])} galaxies")
pixel_cats[example_pix][:5]

HEALPix nside=32: 12288 pixels, ~1.83 deg/pixel side
Assigned 290865 galaxies to 45 non-empty pixels
Built pixel_cats with 45 entries
Galaxies per pixel: min=33, median=8706, max=11427
Example pixel 4812: 11427 galaxies


TARGETID,RA,DEC,PRIORITY,SUBPRIORITY,HEALPIXID
int64,float64,float64,float64,float64,int64
3668,35.74777228679255,11.97390200932178,4.0,0.05188252772662516,4812
4690,35.17375348244834,12.023896459353324,4.0,0.7412575231943688,4812
4711,35.31094616900008,11.738061722171237,4.0,0.38869723041673676,4812
4847,34.375050395082845,12.23613587973983,4.0,0.39297044395520897,4812
4941,35.759695243781124,11.73171700601265,4.0,0.34801973535825614,4812


In [7]:
## save MTL in different fits files with healpix IDs
mtl_dir = out_dir + f"/mtl_nside{nside}/"
os.makedirs(mtl_dir, exist_ok=True)
for pix_id, cat_pix in pixel_cats.items():
    ofile = os.path.join(mtl_dir, f"mtl_healpix_{pix_id:05d}.fits")
    cat_pix.write(ofile, overwrite=True)
print(f"Wrote {len(pixel_cats)} pixel catalogs to {mtl_dir}")

del gal_MTL, pixel_cats
gc.collect()

Wrote 45 pixel catalogs to /home/zjding/fiberassignment/JUST/BGS_mock/Junyu_mock/fba/output/3passes/183tiles_30.0ra40.0_10.0dec20.0/seed3//mtl_nside32/


0

In [8]:
def fba_onetile(tile_ra, tile_dec, tile_id, gal_mtl, neighboring_fiber_pairs, fiberpos_xy):
    """
    tile_ra: float
    tile_dec: float
    tile_id: int
    gal_mtl: galaxy catalog select from healpixID fits files
    
    
    """
    in_tile = mask_targets_in_tile(
        tile_ra, tile_dec, gal_mtl["RA"], gal_mtl["DEC"],
        TILE_INNER_RADIUS_DEG, TILE_OUTER_RADIUS_DEG,
    )
    gal_in_tile = gal_mtl[in_tile]
    if len(gal_in_tile) == 0:
        raise RuntimeError(
            f"Tile {tile_id} ({tile_ra:.4f}, {tile_dec:.4f}): "
            f"no targets in tile annulus"
        )

    target_id_array_unique = np.unique(gal_in_tile["TARGETID"])
    _log(f"  Tile {tile_id}: {len(target_id_array_unique)} unique targets in annulus")


    priority_total = gal_in_tile["PRIORITY"] + gal_in_tile["SUBPRIORITY"]

    tile_key, targets_id_list = find_targets_in_one_tile((
        tile_id, tile_ra, tile_dec, gal_in_tile, fiberpos_xy,
        TILE_INNER_RADIUS_DEG, TILE_OUTER_RADIUS_DEG, R_PATROL_DEG, "TARGETID",
    ))
    targets_id_list_alltiles = {tile_key: targets_id_list}   # just one tile
    
    _, first_idx = np.unique(gal_in_tile["TARGETID"], return_index=True)
    first_rows = gal_in_tile[np.sort(first_idx)]
    target_positions = {
        int(tid): (float(ra), float(dec))
        for tid, ra, dec in zip(first_rows["TARGETID"], first_rows["RA"], first_rows["DEC"])
    }

    tiles_ra = np.asarray([tile_ra], dtype=np.float64)  # to make it as a list in order to use solve_tile_group function
    tiles_dec = np.asarray([tile_dec], dtype=np.float64)
    tiles_id_arr = np.asarray([tile_id], dtype=np.int64)

    collision_constraints = find_collided_pairs_in_one_tile((
        tile_key, 0, tiles_ra, tiles_dec, targets_id_list_alltiles,
        neighboring_fiber_pairs, target_positions, fiberpos_xy,
        COLLISION_SEPARATION_ARCSEC,
    ))
    n_pairs = sum(len(p) for p in collision_constraints.values())
    _log(f"  Tile {tile_id}: {len(collision_constraints)} fiber-pair constraints, {n_pairs} target pairs")

    if eval_workers <= 1:
        with _avoid_nested_pool_in_solve():
            flow_dict, cost, forbidden, n_assigned, n_used_fibers = solve_tile_group(
                target_positions, [0], targets_id_list_alltiles,
                target_id_array_unique, priority_total, collision_constraints,
                COST_OVERFLOW, N_fibers,
                max_iterations=max_iterations,
                n_workers=eval_workers,
                tiles_id=tiles_id_arr,
            )
    else:
        flow_dict, cost, forbidden, n_assigned, n_used_fibers = solve_tile_group(
            target_positions, [0], targets_id_list_alltiles,
            target_id_array_unique, priority_total, collision_constraints,
            COST_OVERFLOW, N_fibers,
            max_iterations=max_iterations,
            n_workers=eval_workers,
            tiles_id=tiles_id_arr,
        )
    _log(
        f"  solve_tile_group finished: "
        f"cost={cost:.2f}, assigned={n_assigned}, used_fibers={n_used_fibers}/{N_fibers}"
    )

    fba_result = aggregate_group_assignments_with_pairwise_repair(
        flow_dict=flow_dict,
        target_ids=target_id_array_unique,
        all_forbidden=set(),
        collision_constraints=collision_constraints,
        gal_cat=gal_in_tile,
        apply_repair=True,
        verbose=True,
    )
    
    return fba_result, targets_id_list_alltiles

In [9]:
# Search radius for loading nearby galaxies (collision scale)
##near_radius_deg = 1.2 * COLLISION_SEPARATION_DEG   ## found a bug
near_radius_deg = 1.2 * TILE_OUTER_RADIUS_DEG
_log(f"Load the sample from healpixID files, where some galaxies are within {near_radius_deg:.6f} deg of the tile center.")

for passid in range(1):
    t0 = time.time()
    _log(f"passid: {passid}")
    mask = (tiles_sub["PASS"] == passid)
    tiles_ra_pass = np.asarray(tiles_sub["RA"][mask], dtype=np.float64)
    tiles_dec_pass = np.asarray(tiles_sub["DEC"][mask], dtype=np.float64)
    tiles_id_pass = np.asarray(tiles_sub["TILEID"][mask], dtype=np.int64)
    _log(f"Number of tiles in pass {passid}: {len(tiles_ra_pass)}")

    for tile_ra, tile_dec, tile_id in zip(tiles_ra_pass[0:2], tiles_dec_pass[0:2], tiles_id_pass[0:2]):
        pix_ids = _healpix_ids_for_disc(tile_ra, tile_dec, near_radius_deg, nside, nest=nest)
        gal_mtl = load_galaxies_from_mtlpix(
            tile_ra,
            tile_dec,
            mtl_dir,
            near_radius_deg,
            nside=nside,
            nest=nest,
        )
        _log(
            f"Tile {tile_id} ({tile_ra:.4f}, {tile_dec:.4f}): "
            f"loaded {len(gal_mtl)} galaxies from {len(pix_ids)} HEALPix file(s) "
            f"pix={pix_ids.tolist()}"
        )

        t_fba_start = time.time()
        # output targets_id_list_alltiles for write_fba_onetile function
        fba_result, targets_id_list_alltiles = fba_onetile(tile_ra, tile_dec, tile_id, gal_mtl, neighboring_fiber_pairs, fiberpos_xy)
        t_fba_end = time.time()
        _log(f"  fba process finished in {t_fba_end - t_fba_start:.2f}s")
        
        per_tile_assigned = defaultdict(lambda: {"target_ids": [], "fiber_ids": []})
        for _tid, _node in fba_result["target_to_fiber"].items():
            _node = str(_node)
            if "_fiber_" not in _node:
                continue
            _tile, _fid = _node.rsplit("_fiber_", 1)
            try:
                _fid = int(_fid)
            except Exception:
                continue
            per_tile_assigned[_tile]["target_ids"].append(int(_tid))
            per_tile_assigned[_tile]["fiber_ids"].append(_fid)
    
        assigned_tarids_all = []
        for _tile, _vals in per_tile_assigned.items():
            assigned_targets_id = _vals["target_ids"]
            assigned_tarids_all.extend(assigned_targets_id)
            out_fits = _fits_path_for_tile(out_dir, tile_id)
            write_fba_onetile(
                tile_id=_tile,
                assigned_targets_id=assigned_targets_id,
                assigned_fiber_id=_vals["fiber_ids"],
                targets_id_list_alltiles=targets_id_list_alltiles,
                out_fits_path=out_fits,
                overwrite=True,
            )
        t_outfits_end = time.time()
        _log(f"  Wrote {out_fits} ({len(assigned_targets_id)} assignments); takes {t_outfits_end - t_fba_end:.2f}s")
    
        if assigned_tarids_all:
            n_disk = _degrade_mtl_priorities_on_disk(
                mtl_dir, assigned_tarids_all, priority_degraded, pix_ids
            )
            _log(
                f"  Updated PRIORITY on disk for {n_disk} row(s) "
                f"in {len(pix_ids)} pixel file(s): {pix_ids.tolist()}"
            )
        _log(f"  Update the MTL PRIORITY files in {time.time() - t_outfits_end:.2f}s\n")
        _log("=====================")

Load the sample from healpixID files, where some galaxies are within 0.716160 deg of the tile center.
passid: 0
Number of tiles in pass 0: 61
Tile 100408 (39.4021, 17.0056): loaded 29479 galaxies from 4 HEALPix file(s) pix=[4174, 4301, 4302, 4430]
  Tile 100408: 3297 unique targets in annulus
  Tile 100408: 119 fiber-pair constraints, 183 target pairs
  solve_tile_group finished: cost=14028.26, assigned=1605, used_fibers=1605/2184

=== Single Tile/Group Complete ===
All constraints satisfied.
  fba process finished in 52.29s
  Wrote /home/zjding/fiberassignment/JUST/BGS_mock/Junyu_mock/fba/output/3passes/183tiles_30.0ra40.0_10.0dec20.0/seed3/fba_tile_100408.fits (1605 assignments); takes 0.02s
  Updated PRIORITY on disk for 1605 row(s) in 4 pixel file(s): [4174, 4301, 4302, 4430]
  Update the MTL PRIORITY files in 0.06s

Tile 100409 (38.6536, 17.9527): loaded 37396 galaxies from 4 HEALPix file(s) pix=[4045, 4173, 4174, 4301]
  Tile 100409: 3119 unique targets in annulus
  Tile 100409: 

In [10]:
ifile = "/home/zjding/fiberassignment/JUST/BGS_mock/Junyu_mock/fba/output/3passes/183tiles_30.0ra40.0_10.0dec20.0/seed3/fba_tile_100409.fits"
cat_assigned = Table.read(ifile, hdu=1)
cat_assigned

TARGETID,FIBERID
int64,int32
346069701,0
258312281,2
302181527,3
431055911,4
258312635,6
171691044,7
128295751,8
431053912,9
518260413,11


In [11]:
cat_avail = Table.read(ifile, hdu=2)
cat_avail

TARGETID,FIBERID
int64,int32
302185375,0
346069701,0
215100857,2
258312281,2
873002324,2
890896345,2
917409566,2
917409587,2
87126226,3
